# Análisis de Valores de Shapley (Dinámico)

Este notebook implementa el análisis de valores de Shapley (Shapley values) para cuantificar la contribución relativa de las variables de comportamiento al poder predictivo del modelo GAM, similar al análisis de la Figura 5 del paper `bats.pdf`. Esta versión soporta N variables dinámicamente.

In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append(os.path.abspath('.'))

from scripts.utils.data_loader import (
    preparar_datos_posicion,
    preparar_datos_head_direction,
    preparar_datos_mirada,
)
from scripts.utils.cross_validation import generate_all_splits
from scripts.utils.shapley_values import train_and_evaluate_shapley_subsets, calculate_shapley_values, plot_shapley_values

In [ ]:
sesion = 2
tetrodo = 3
neurona = 1
bin_size_sec = 0.1

In [ ]:
# Cargar datos básicos de head direction
ang_bins_deg, spikes_hd = preparar_datos_head_direction(sesion, tetrodo, neurona, bin_size_sec=bin_size_sec)
hd_rad = np.radians(ang_bins_deg)

# Cargar datos de posición y mirada
x_bins, y_bins, ang_bins_rad, spikes_gaze = preparar_datos_mirada(
    sesion, tetrodo, neurona, bin_size_sec=bin_size_sec
)

# Punto de interés óptimo (usamos uno precalculado como ejemplo)
x_best_p, y_best_p = 17.7, -5.2
distances = np.sqrt((x_bins - x_best_p)**2 + (y_bins - y_best_p)**2)

# Viewpoint: ángulo relativo hacia el punto de interés
dy = y_best_p - y_bins
dx = x_best_p - x_bins
ang_hacia_punto = np.arctan2(dy, dx)
theta_corrected = np.mod(ang_bins_rad - ang_hacia_punto, 2 * np.pi)

# Crear matriz de diseño X de tamaño (N, 5)
# Columnas: 0=Posición X, 1=Posición Y, 2=Head Direction (rad), 3=Viewpoint (rad), 4=Distancia
X_shapley = np.column_stack((x_bins, y_bins, hd_rad, theta_corrected, distances))
Y_shapley = spikes_gaze

print(f"Total de muestras: {len(X_shapley)}")

# Partición de datos con el bug de distorsión temporal corregido
folds, held_out_idx, train_pool_idx, roles = generate_all_splits(
    n_muestras=len(X_shapley),
    bin_size_sec=bin_size_sec,
    block_size_sec=60,
    n_folds=5,
    buffer_sec=2
)

X_pool, Y_pool = X_shapley[train_pool_idx], Y_shapley[train_pool_idx]
X_held, Y_held = X_shapley[held_out_idx], Y_shapley[held_out_idx]

In [ ]:
# Lista de variables a evaluar con Shapley
# Puedes agregar o quitar variables de esta lista libremente.
features = ['pos', 'hd', 'view', 'dist']

# Entrenamos todos los subconjuntos (2^N - 1 = 15 modelos para 4 variables)
metrics = train_and_evaluate_shapley_subsets(
    X_shapley, Y_shapley, folds, X_pool, Y_pool, X_held, Y_held,
    splines_grid=[9], lambdas_grid=np.logspace(-1, 1, 7), features=features
)

In [ ]:
# Calcular contribuciones Shapley
phi = calculate_shapley_values(metrics, features=features)

# Visualizar
total_r2 = metrics['shapley_' + '_'.join(features)] - metrics['null']
plot_shapley_values(phi, total_r2=total_r2)